# 1. Importação dos Dados e Bibliotecas.

Nesta etapa, a biblioteca Pandas é importada e o arquivo de assinaturas é carregado e convertido em um DataFrame.

In [1]:
from pathlib import Path

RAIZ    = Path.cwd().parent
BRONZE  = RAIZ / "data" / "Bronze"
STAGING = RAIZ / "data" / "Staging"
SILVER  = RAIZ / "data" / "Silver"

assert BRONZE.exists(), f"Pasta não encontrada: {BRONZE}. Abra o notebook a partir da pasta notebooks/."

STAGING.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)

# Importação de Biblioteca:
import pandas as pd
import numpy as np

df_planos = pd.read_csv(BRONZE/'planos.csv')
print(df_planos.info())

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_plano      8 non-null      int64  
 1   nome_plano    8 non-null      str    
 2   categoria     8 non-null      str    
 3   valor_mensal  7 non-null      float64
dtypes: float64(1), int64(1), str(2)
memory usage: 527.0 bytes
None


# 2. Tratamento e Análise de Dados:

Nesta etapa, é realizado o tratamento completo dos dados, incluindo a validação de registros duplicados, a alteração de status e a conversão de tipos (types), com base na análise dos seguintes pontos críticos;

## 2.1 Tratamento da Coluna id_plano

Nesta etapa, será realizada a tratativa da coluna id_plano, a fim de validar se existem duplicadas e além disso, é feito a remocação também de duplicadas da planilha em geral.

In [2]:
# Remoção e Validação de linhas duplicadas dentro de df:
# 1. Validação pré-tratamento: Conta quantas linhas totalmente duplicadas existem na tabela inteira
total_linhas_duplicadas = df_planos.duplicated().sum()
print(f"Total de linhas duplicadas: {total_linhas_duplicadas}")

# 1.2. Tratamento: Remove as linhas duplicadas mantendo apenas a primeira ocorrência
df_planos = df_planos.drop_duplicates(keep='first').reset_index(drop=True)

# 2. Validação pós-tratamento: Nova contagem para garantir que ficou zerado
total_linhas_duplicadas_pos = df_planos.duplicated().sum()
print(f"Total de linhas duplicadas pós tratamento: {total_linhas_duplicadas_pos}")

Total de linhas duplicadas: 0
Total de linhas duplicadas pós tratamento: 0


## 2.2 Tratamento da coluna nome_plano.

Nesta etapa, será realizada a tratativa de dados e padronização da coluna de nome_planos, garantindo que os dados estejam corretos para o relatório final.

In [3]:
# 1. Identificando valores contidos na coluna para tratamento de dados.
valores_unicos = df_planos['nome_plano'].unique()
print(f'Dados brutos (Sem Tratamento): {valores_unicos}')

# 2. Realizando a correção dos nomes e padronização dos dados:
# 2.1 Removendo todos os espaços extras de dados:
df_planos['nome_plano'] = df_planos['nome_plano'].str.strip()

# 2.2 Definindo todos os valores válidos (planos e estados permitidos)
valores_validos = {
    'Básico', 'Standard', 'Premium', 'Família', 
    'Start', 'Plus', 'Business', 'Enterprise'
}

# 2.3 Condicional: Se o valor não estiver na lista de válidos, é modificado para "Investigar"
df_planos['nome_plano'] = df_planos['nome_plano'].apply(
    lambda x: x if x in valores_validos else 'Investigar'
)

# 3. Validação de Dados Pós Tratamento:
valores_unicos_tratados = df_planos['nome_plano'].unique()
print(f'Pós Tratamento: {valores_unicos_tratados}')

Dados brutos (Sem Tratamento): <ArrowStringArray>
[    'Básico',   'Standard',    'Premium',    'Família',      'Start',
       'Plus',   'Business', 'Enterprise']
Length: 8, dtype: str
Pós Tratamento: <ArrowStringArray>
[    'Básico',   'Standard',    'Premium',    'Família',      'Start',
       'Plus',   'Business', 'Enterprise']
Length: 8, dtype: str


## 2.3 Tratamento da coluna categoria.

Nesta etapa, será realizada a tratativa de dados e padronização da coluna de nome_planos, garantindo que os dados estejam corretos para o relatório final.

In [4]:
# 1. Identificando valores contidos na coluna para tratamento de dados.
valores_unicos = df_planos['categoria'].unique()
print(f'Dados brutos (Sem Tratamento): {valores_unicos}')

# 2. Realizando a correção dos nomes e padronização dos dados:
# 2.1 Removendo todos os espaços extras de dados:
df_planos['categoria'] = df_planos['categoria'].str.strip().str.capitalize()

# 2.2 Condicional: Caso entre algum valor que não esteja nessa lista, será modificado para "Investigar"
# O .get() procura a chave; se não achar, retorna "Investigar" como padrão
categorias_validas = {
    'Individual',
    'Família',
    'Empresarial'
}

df_planos['categoria'] = df_planos['categoria'].apply(
    lambda x: x if x in categorias_validas else 'Investigar'
)

# 3. Validação de Dados Pós Tratamento:
valores_unicos_tratados = df_planos['categoria'].unique()
print(f'Pós Tratamento: {valores_unicos_tratados}')

Dados brutos (Sem Tratamento): <ArrowStringArray>
['Individual', ' individual ', 'Família', 'Empresarial']
Length: 4, dtype: str
Pós Tratamento: <ArrowStringArray>
['Individual', 'Família', 'Empresarial']
Length: 3, dtype: str


## 2.4 Tratamento da coluna valor_mensal

Nesta etapa, procedeu-se à validação, tratamento e correção dos dados da base.

Observação importante: Foi identificado um valor ausente (NaN) na coluna correspondente. Para corrigir esta inconsistência, será necessária uma investigação adicional para determinar e inserir o valor correto correspondente ao plano PLUS (ID do Plano: 6, Categoria: INDIVIDUAL).

In [5]:
# 1. Validação: Retorna True se existir algum número negativo ou igual a zero
tem_anomalias = (df_planos['valor_mensal'] <= 0).any()
print(f'Existem valores negativos ou iguais a zero em valor_mensal? {tem_anomalias}')

# 2. Tratamento: Transforma números negativos em positivos (o .abs() pega o valor absoluto)
df_planos['valor_mensal'] = df_planos['valor_mensal'].abs()
print('Tratamento de valor_mensal concluído com sucesso!')

Existem valores negativos ou iguais a zero em valor_mensal? False
Tratamento de valor_mensal concluído com sucesso!


# 3. Data Quality & Validação.

Nesta etapa, o DataFrame tratado é submetido a testes de validação antes de ser promovido para a camada Staging (Preparação de Dados)

In [6]:
resultado_qualidade = {}

# 1. Validaçaõ de IDs não nulos:
resultado_qualidade['ID não nulo'] = (
    df_planos['id_plano'].isnull().sum() == 0
)

# 2. Validação a fim de identificar se existem IDs duplicados
resultado_qualidade['ID único'] = (
    df_planos['id_plano'].duplicated().sum() == 0
)

# 3. Validação se existem linhas duplicadas
resultado_qualidade['Sem duplicatas completas'] = (
    df_planos.duplicated().sum() == 0
)

# 4. Validação se todas as linhas da coluna id_plano estão preenchidas
resultado_qualidade['Linhas preenchidas id'] = (
    df_planos['id_plano'].notna().all()
)

# 5. Validação da coluna valor_mensal

# NOTA: O teste de qualidade para 'valor_mensal' foi temporariamente desativado.
# Motivo: Existe um valor NaN pendente de investigação e preenchimento pela 
# equipa responsável. Assim que o dado for corrigido, a trava de segurança 
# para números negativos e nulos será reativada.

# CÓDIGO FUTURO (A reativar após limpeza):
# resultado_qualidade['valor_mensal_valido'] = (
# df_planos['valor_mensal'].notna() & (df_planos['valor_mensal'] > 0)).all()

# Contagem de valores ausentes (NaN) para fins de registro
sem_valor = df_planos['valor_mensal'].isna().sum()
print(f"AVISO: {sem_valor} plano(s) sem valor_mensal")

# 6. Validando se restou algum valor indesejado na coluna categoria:
valores_permitidos_categoria = ['Individual', 'Família', 'Empresarial']

# Atribui o resultado True/False diretamente na coluna do dicionário de qualidade
resultado_qualidade['Sem dados instrusos na coluna categoria'] = (
    df_planos['categoria'].isin(valores_permitidos_categoria).all()
)

# 7. Validando se restou algum valor indesejado na coluna nome_plano:
valores_permitidos_nome_plano = [
    'Básico', 'Standard', 'Premium', 'Família', 
    'Start', 'Plus', 'Business', 'Enterprise']

# Atribui o resultado True/False diretamente na coluna do dicionário de qualidade
resultado_qualidade['Sem dados instrusos na coluna nome_plano'] = (
    df_planos['nome_plano'].isin(valores_permitidos_nome_plano).all()
)

AVISO: 1 plano(s) sem valor_mensal


# 4. Data Quality: Homologação e Envio para a Camada Staging (Preparação de Dados)

Nesta etapa, o console exibe o resultado das validações realizadas:

Aprovado (True em todos os testes): O DataFrame é liberado e enviado para a camada Staging (Preparação de Dados)
Reprovado (qualquer valor False): O processo é bloqueado, exigindo a correção dos dados antes de avançar.

In [7]:
df_qualidade = pd.DataFrame(
    list(resultado_qualidade.items()),
    columns=['Regra', 'Aprovado']
)

df_qualidade['Status'] = df_qualidade['Aprovado'].map({
    True: 'APROVADO',
    False: 'REPROVADO'
})

display(df_qualidade)

todos_testes_ok = df_qualidade['Aprovado'].all()
caminho_saida = (STAGING/'dim_planos.csv')

if todos_testes_ok:
    print("Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Camada Staging (Preparação de Dados).")

    df_planos.to_csv(
        caminho_saida,
        index=False,
        sep=';',
        encoding='utf-8-sig'
    )

else:
    print("Alerta: O arquivo foi impedido de subir para a camada Camada Staging (Preparação de Dados). Existem regras reprovadas.")
    raise ValueError("Falha na validação de qualidade dos dados. Processo interrompido.")

,Regra,Aprovado,Status
0,ID não nulo,True,APROVADO
1,ID único,True,APROVADO
2,Sem duplicatas completas,True,APROVADO
3,Linhas preenchidas id,True,APROVADO
4,Sem dados instrusos na coluna categoria,True,APROVADO
5,Sem dados instrusos na coluna nome_plano,True,APROVADO


Sucesso: Todos os testes passaram. O arquivo foi enviado para a camada Camada Staging (Preparação de Dados).
